# STIXBert: Self-Supervised STIX Graph Foundation Model

**Goal**: Pre-train a Heterogeneous Graph Transformer (HGT) on STIX 2.1
threat intelligence graphs using three self-supervised objectives:
1. Masked node prediction (GraphMAE-style)
2. Link prediction
3. Temporal ordering

Then demonstrate downstream tasks: campaign clustering, ATT&CK classification,
cross-feed deduplication.

**Runtime**: Select GPU (T4 for dev, A100 for full training)

> **Two versions of this project exist:**
> - `src/` — importable Python package for local dev & production
> - This notebook — fully self-contained for Colab (no imports from `src/`)

## 1. Setup — Install, Mount Drive, Configure

In [ ]:
# Install dependencies
!pip install -q torch-geometric stix2 taxii2-client sentence-transformers \
    scikit-learn umap-learn matplotlib hdbscan huggingface_hub

import os, sys, json, random, logging
from pathlib import Path
from collections import defaultdict
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()} "
      f"({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'})")

# --- Mount Google Drive (data, checkpoints, results persist here) ---
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/stixbert')
PATHS = {
    'raw_mitre':       DRIVE_ROOT / 'data/raw/mitre_attack',
    'raw_threatfox':   DRIVE_ROOT / 'data/raw/threatfox',
    'raw_digitalside': DRIVE_ROOT / 'data/raw/digitalside',
    'processed':       DRIVE_ROOT / 'data/processed',
    'train':           DRIVE_ROOT / 'data/train',
    'test':            DRIVE_ROOT / 'data/test',
    'eval':            DRIVE_ROOT / 'data/eval',
    'checkpoints':     DRIVE_ROOT / 'checkpoints',
    'results':         DRIVE_ROOT / 'results',
}
for p in PATHS.values():
    p.mkdir(parents=True, exist_ok=True)

# --- Configuration (single dict — no external files needed) ---
CFG = {
    'model': {
        'name': 'stixbert', 'version': '0.1.0',
        'input_dim': 128, 'hidden_dim': 128, 'output_dim': 128,
        'num_heads': 4, 'num_layers': 4, 'dropout': 0.1,
        'text_model': 'all-MiniLM-L6-v2',
    },
    'training': {
        'num_epochs': 200, 'lr': 1e-3, 'weight_decay': 1e-4,
        'mask_ratio': 0.15, 'grad_clip_norm': 1.0,
        'checkpoint_every': 20, 'log_every': 5,
        'loss_weights': {'masked_node': 1.0, 'link_pred': 1.0, 'temporal': 0.5},
    },
    'finetune': {
        'label_efficiency_fractions': [0.01, 0.05, 0.1, 0.25, 0.5, 1.0],
    },
    'huggingface': {
        'repo_id': 'cisco-sbg/stixbert', 'private': True,
        'tags': ['threat-intelligence', 'stix', 'graph-neural-network',
                 'hgt', 'cybersecurity'],
    },
}

# --- Reproducibility ---
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(name)s %(levelname)s: %(message)s',
)
logger = logging.getLogger('stixbert')

print(f"\nDrive root: {DRIVE_ROOT}")
print(f"Device: {DEVICE} | Seed: {SEED}")
m = CFG['model']
print(f"Model: {m['hidden_dim']}d, {m['num_heads']} heads, {m['num_layers']} layers")
print(f"Pre-train: {CFG['training']['num_epochs']} epochs, lr={CFG['training']['lr']}")

## 2. Data Retrieval — MITRE ATT&CK, ThreatFox, DigitalSide

In [ ]:
import urllib.request

# ── MITRE ATT&CK ────────────────────────────────────────────────────
ATTACK_URLS = {
    'enterprise': 'https://raw.githubusercontent.com/mitre-attack/attack-stix-data/master/enterprise-attack/enterprise-attack.json',
    'mobile':     'https://raw.githubusercontent.com/mitre-attack/attack-stix-data/master/mobile-attack/mobile-attack.json',
    'ics':        'https://raw.githubusercontent.com/mitre-attack/attack-stix-data/master/ics-attack/ics-attack.json',
}

def download_attack_bundle(domain, output_dir):
    dest = output_dir / f'{domain}-attack.json'
    if dest.exists():
        logger.info(f'Using cached: {dest}')
        with open(dest) as f: return json.load(f)
    logger.info(f'Downloading ATT&CK {domain}...')
    urllib.request.urlretrieve(ATTACK_URLS[domain], dest)
    with open(dest) as f: return json.load(f)

bundles = []
for domain in ['enterprise', 'mobile', 'ics']:
    b = download_attack_bundle(domain, PATHS['raw_mitre'])
    bundles.append(b)
    types = {}
    for obj in b.get('objects', []):
        t = obj.get('type', 'unknown')
        types[t] = types.get(t, 0) + 1
    total = sum(types.values())
    print(f"\nATT&CK {domain}: {total} objects")
    for t, c in sorted(types.items(), key=lambda x: -x[1])[:8]:
        print(f"  {t}: {c}")

# ── ThreatFox ────────────────────────────────────────────────────────
# NOTE: The POST API (threatfox-api.abuse.ch/api/v1/) now requires auth.
# Using the public export endpoint instead.
THREATFOX_EXPORT_URL = 'https://threatfox.abuse.ch/export/json/recent/'

def fetch_threatfox_iocs(days=30):
    logger.info(f'Fetching ThreatFox recent IOCs (export endpoint)...')
    req = urllib.request.Request(THREATFOX_EXPORT_URL)
    with urllib.request.urlopen(req) as resp:
        raw = json.loads(resp.read())
    # Export format: dict keyed by IOC id -> IOC record.
    iocs = list(raw.values()) if isinstance(raw, dict) else raw
    # Flatten: each value may itself be a dict with nested fields.
    flat_iocs = []
    for item in iocs:
        if isinstance(item, dict):
            flat_iocs.append(item)
        elif isinstance(item, list):
            flat_iocs.extend(item)
    out = PATHS['raw_threatfox'] / 'threatfox_recent.json'
    with open(out, 'w') as f: json.dump(flat_iocs, f, indent=2)
    logger.info(f'ThreatFox: fetched {len(flat_iocs)} IOCs')
    return flat_iocs

def threatfox_to_stix(iocs):
    from stix2 import Bundle, Indicator, Malware, Relationship
    stix_objects, malware_cache = [], {}
    for ioc in iocs:
        ioc_type = ioc.get('ioc_type', '')
        ioc_value = ioc.get('ioc', '')
        malware_name = ioc.get('malware_printable', '')
        threat_type = ioc.get('threat_type', '')
        pattern = None
        if 'ip' in ioc_type.lower():
            pattern = f"[ipv4-addr:value = '{ioc_value.split(':')[0]}']"
        elif 'domain' in ioc_type.lower():
            pattern = f"[domain-name:value = '{ioc_value}']"
        elif 'url' in ioc_type.lower():
            pattern = f"[url:value = '{ioc_value}']"
        elif 'md5' in ioc_type.lower():
            pattern = f"[file:hashes.MD5 = '{ioc_value}']"
        elif 'sha256' in ioc_type.lower():
            pattern = f"[file:hashes.'SHA-256' = '{ioc_value}']"
        if not pattern:
            continue
        indicator = Indicator(
            name=f'ThreatFox: {ioc_value}', pattern=pattern, pattern_type='stix',
            valid_from=ioc.get('first_seen_utc', datetime.now().isoformat()),
            labels=[threat_type] if threat_type else ['malicious-activity'],
        )
        stix_objects.append(indicator)
        if malware_name and malware_name not in malware_cache:
            mw = Malware(name=malware_name, is_family=True,
                         malware_types=['unknown'])
            malware_cache[malware_name] = mw
            stix_objects.append(mw)
        if malware_name and malware_name in malware_cache:
            stix_objects.append(Relationship(
                source_ref=indicator.id,
                target_ref=malware_cache[malware_name].id,
                relationship_type='indicates',
            ))
    return json.loads(Bundle(objects=stix_objects).serialize())

threatfox_iocs = fetch_threatfox_iocs()
threatfox_stix = threatfox_to_stix(threatfox_iocs)
print(f'\nThreatFox -> STIX: {len(threatfox_stix.get("objects", []))} objects')

# ── DigitalSide ──────────────────────────────────────────────────────
# The stix2/ directory contains ~1000 individual hash-named STIX bundles.
# We use the GitHub API to list files and download a batch.
DIGITALSIDE_API = ('https://api.github.com/repos/davidonzo/'
                   'Threat-Intel/contents/stix2')
DIGITALSIDE_RAW = ('https://raw.githubusercontent.com/'
                   'davidonzo/Threat-Intel/master/stix2/')
MAX_DIGITALSIDE_BUNDLES = 50  # limit to keep runtime reasonable

def fetch_digitalside_stix(max_bundles=MAX_DIGITALSIDE_BUNDLES):
    logger.info('Listing DigitalSide stix2/ via GitHub API...')
    req = urllib.request.Request(DIGITALSIDE_API,
                                headers={'Accept': 'application/vnd.github.v3+json'})
    with urllib.request.urlopen(req) as resp:
        listing = json.loads(resp.read())
    json_files = [f['name'] for f in listing
                  if isinstance(f, dict) and f.get('name', '').endswith('.json')]
    logger.info(f'Found {len(json_files)} STIX bundles; downloading {max_bundles}')
    bundles = []
    for fname in json_files[:max_bundles]:
        url = f'{DIGITALSIDE_RAW}{fname}'
        dest = PATHS['raw_digitalside'] / fname
        try:
            urllib.request.urlretrieve(url, dest)
            with open(dest) as f: bundle = json.load(f)
            bundles.append(bundle)
        except Exception as e:
            logger.warning(f'Failed: {fname}: {e}')
    logger.info(f'DigitalSide: loaded {len(bundles)} STIX bundles')
    return bundles

digitalside_bundles = fetch_digitalside_stix()
total_ds_objs = sum(len(b.get('objects', [])) for b in digitalside_bundles)
print(f'DigitalSide: {len(digitalside_bundles)} bundles, '
      f'{total_ds_objs} total objects')

print(f'\nAll raw data cached under: {DRIVE_ROOT}/data/raw/')

## 3. Build Heterogeneous Graph

In [ ]:
from torch_geometric.data import HeteroData

# ── Edge / node type constants ───────────────────────────────────────
EDGE_TYPE_MAP = {
    'uses': 'uses', 'indicates': 'indicates', 'targets': 'targets',
    'attributed-to': 'attributed_to',
    'communicates-with': 'communicates_with',
    'exploits': 'exploits', 'mitigates': 'mitigates',
    'derived-from': 'derived_from', 'related-to': 'related_to',
    'consists-of': 'consists_of', 'controls': 'controls',
    'hosts': 'hosts', 'located-at': 'located_at',
    'originates-from': 'originates_from', 'delivers': 'delivers',
    'drops': 'drops', 'variant-of': 'variant_of',
}
SDO_TYPES = {
    'indicator', 'malware', 'attack-pattern', 'threat-actor', 'campaign',
    'intrusion-set', 'infrastructure', 'vulnerability', 'tool', 'identity',
    'location', 'course-of-action',
}

def normalize_type(t):
    return t.replace('-', '_')


class STIXGraphBuilder:
    def __init__(self, include_scos=False):
        self.allowed = SDO_TYPES.copy()
        self.node_registry = {}
        self.nodes_by_type = defaultdict(list)
        self.edges = defaultdict(list)
        self._rels, self._sightings = [], []

    def add_bundle(self, bundle, source_name='unknown'):
        for obj in bundle.get('objects', []):
            t = obj.get('type', '')
            if t == 'relationship':
                self._rels.append(obj)
            elif t == 'sighting':
                self._sightings.append(obj)
            elif t in self.allowed:
                self._register(obj, source_name)
        self._process_rels()
        self._process_sightings()

    def _register(self, obj, source):
        nt, sid = normalize_type(obj['type']), obj['id']
        if sid in self.node_registry:
            return
        idx = len(self.nodes_by_type[nt])
        self.node_registry[sid] = (nt, idx)
        obj['_source'] = source
        self.nodes_by_type[nt].append(obj)

    def _process_rels(self):
        for rel in self._rels:
            s = rel.get('source_ref', '')
            d = rel.get('target_ref', '')
            if s not in self.node_registry or d not in self.node_registry:
                continue
            rt = rel.get('relationship_type', '')
            et = EDGE_TYPE_MAP.get(rt, rt.replace('-', '_'))
            snt, si = self.node_registry[s]
            dnt, di = self.node_registry[d]
            self.edges[(snt, et, dnt)].append((si, di))
        self._rels.clear()

    def _process_sightings(self):
        for s in self._sightings:
            sof = s.get('sighting_of_ref', '')
            if sof not in self.node_registry:
                continue
            st, si = self.node_registry[sof]
            for ref in s.get('where_sighted_refs', []):
                if ref in self.node_registry:
                    dt, di = self.node_registry[ref]
                    self.edges[(st, 'sighted_by', dt)].append((si, di))
        self._sightings.clear()

    def build(self, feature_dim=128):
        hetero = HeteroData()
        for nt, nodes in self.nodes_by_type.items():
            n = len(nodes)
            hetero[nt].x = torch.randn(n, feature_dim)
            hetero[nt].num_nodes = n
        for (st, et, dt), el in self.edges.items():
            if not el:
                continue
            si, di = zip(*el)
            hetero[st, et, dt].edge_index = torch.tensor(
                [list(si), list(di)], dtype=torch.long,
            )
        return hetero

    def get_stats(self):
        return {
            'total_nodes': sum(len(v) for v in self.nodes_by_type.values()),
            'total_edges': sum(len(v) for v in self.edges.values()),
            'node_types': {k: len(v) for k, v in self.nodes_by_type.items()},
            'edge_types': {
                f'({s},{e},{d})': len(v)
                for (s, e, d), v in self.edges.items()
            },
        }


# ── Build the graph ──────────────────────────────────────────────────
builder = STIXGraphBuilder(include_scos=False)
for i, b in enumerate(bundles):
    builder.add_bundle(
        b, source_name=f'mitre_{["enterprise", "mobile", "ics"][i]}',
    )
builder.add_bundle(threatfox_stix, source_name='threatfox')
for b in digitalside_bundles:
    builder.add_bundle(b, source_name='digitalside')

stats = builder.get_stats()
print(f"Graph: {stats['total_nodes']} nodes, {stats['total_edges']} edges")
print(f"\nNode types:")
for nt, c in sorted(stats['node_types'].items(), key=lambda x: -x[1]):
    print(f"  {nt}: {c}")
print(f"\nEdge types:")
for et, c in sorted(stats['edge_types'].items(), key=lambda x: -x[1]):
    print(f"  {et}: {c}")

with open(PATHS['processed'] / 'graph_stats.json', 'w') as f:
    json.dump(stats, f, indent=2)

In [ ]:
from sentence_transformers import SentenceTransformer


class FeatureEncoder:
    def __init__(self, text_model_name='all-MiniLM-L6-v2',
                 output_dim=128, device='cpu'):
        self.output_dim = output_dim
        self.device = device
        self._model = SentenceTransformer(text_model_name)
        logger.info(
            f"Loaded '{text_model_name}' "
            f"(dim={self._model.get_sentence_embedding_dimension()})",
        )

    def encode_stix_nodes(self, nodes):
        texts = []
        for n in nodes:
            name = n.get('name', '')
            desc = n.get('description', '')
            t = f'{name}. {desc}' if desc else name
            texts.append(t if t.strip() else n.get('id', 'unknown'))
        emb = self._model.encode(
            texts, show_progress_bar=True, convert_to_tensor=True,
        ).to(self.device)
        proj = nn.Linear(emb.shape[1], self.output_dim).to(self.device)
        with torch.no_grad():
            return proj(emb)

    def encode_all(self, nodes_by_type):
        encoded = {}
        for nt, nodes in nodes_by_type.items():
            logger.info(f'Encoding {len(nodes)} "{nt}" nodes...')
            encoded[nt] = self.encode_stix_nodes(nodes)
        return encoded


# ── Encode & build HeteroData ────────────────────────────────────────
feat_encoder = FeatureEncoder(
    text_model_name=CFG['model']['text_model'],
    output_dim=CFG['model']['input_dim'],
    device=DEVICE,
)
encoded_features = feat_encoder.encode_all(builder.nodes_by_type)

data = builder.build(feature_dim=CFG['model']['input_dim'])
for nt, feat in encoded_features.items():
    data[nt].x = feat.cpu()

print(f'\nHeteroData: {data}')
print(f'Node types: {data.node_types}')
print(f'Edge types: {data.edge_types}')

torch.save(data, PATHS['processed'] / 'hetero_data.pt')
print(f"Saved to: {PATHS['processed'] / 'hetero_data.pt'}")

## 4. STIXBert Model

In [ ]:
from torch_geometric.nn import HGTConv, Linear as PyGLinear


class STIXBertEncoder(nn.Module):
    def __init__(self, node_types, edge_types, input_dim=128,
                 hidden_dim=128, output_dim=128, num_heads=4,
                 num_layers=4, dropout=0.1):
        super().__init__()
        self.input_proj = nn.ModuleDict(
            {nt: PyGLinear(input_dim, hidden_dim) for nt in node_types},
        )
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            self.convs.append(HGTConv(
                hidden_dim, hidden_dim,
                metadata=(node_types, edge_types), heads=num_heads,
            ))
            self.norms.append(nn.ModuleDict(
                {nt: nn.LayerNorm(hidden_dim) for nt in node_types},
            ))
        self.output_proj = nn.ModuleDict(
            {nt: PyGLinear(hidden_dim, output_dim) for nt in node_types},
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x_dict, edge_index_dict):
        h = {
            nt: self.input_proj[nt](x)
            for nt, x in x_dict.items() if nt in self.input_proj
        }
        for i, conv in enumerate(self.convs):
            h_new = conv(h, edge_index_dict)
            h = {
                nt: self.norms[i][nt](self.dropout(h_new.get(nt, v)) + v)
                for nt, v in h.items()
            }
        return {
            nt: self.output_proj[nt](v)
            for nt, v in h.items() if nt in self.output_proj
        }


class MaskedNodeHead(nn.Module):
    def __init__(self, node_types, dim=128):
        super().__init__()
        self.dec = nn.ModuleDict({
            nt: nn.Sequential(
                PyGLinear(dim, dim), nn.GELU(), PyGLinear(dim, dim),
            )
            for nt in node_types
        })

    def forward(self, emb, orig, masks):
        total, count = 0.0, 0
        for nt in emb:
            if nt not in masks or masks[nt].sum() == 0:
                continue
            m = masks[nt]
            pred = self.dec[nt](emb[nt][m])
            target = orig[nt][m]
            total += (1 - F.cosine_similarity(pred, target, dim=-1)).mean()
            count += 1
        return total / max(count, 1)


class LinkPredHead(nn.Module):
    def __init__(self, dim=128):
        super().__init__()
        self.net = nn.Sequential(
            PyGLinear(dim * 2, dim), nn.GELU(), PyGLinear(dim, 1),
        )

    def forward(self, src, dst):
        return self.net(torch.cat([src, dst], dim=-1)).squeeze(-1)


class TemporalHead(nn.Module):
    def __init__(self, dim=128):
        super().__init__()
        self.net = nn.Sequential(
            PyGLinear(dim * 2, dim), nn.GELU(), PyGLinear(dim, 1),
        )

    def forward(self, a, b):
        return self.net(torch.cat([a, b], dim=-1)).squeeze(-1)


class STIXBert(nn.Module):
    def __init__(self, node_types, edge_types, input_dim=128,
                 hidden_dim=128, num_heads=4, num_layers=4, dropout=0.1):
        super().__init__()
        self.encoder = STIXBertEncoder(
            node_types, edge_types, input_dim, hidden_dim, hidden_dim,
            num_heads, num_layers, dropout,
        )
        self.masked_node_head = MaskedNodeHead(node_types, hidden_dim)
        self.link_pred_head = LinkPredHead(hidden_dim)
        self.temporal_head = TemporalHead(hidden_dim)

    def forward(self, x_dict, edge_index_dict):
        return self.encoder(x_dict, edge_index_dict)

    def get_embeddings(self, x_dict, edge_index_dict):
        self.eval()
        with torch.no_grad():
            return self.encoder(x_dict, edge_index_dict)


class ATTACKClassifier(nn.Module):
    def __init__(self, input_dim, num_classes, dropout=0.1):
        super().__init__()
        self.clf = nn.Sequential(
            PyGLinear(input_dim, input_dim), nn.GELU(),
            nn.Dropout(dropout), PyGLinear(input_dim, num_classes),
        )

    def forward(self, x):
        return self.clf(x)


class CampaignClusterer:
    def __init__(self, method='hdbscan', n_clusters=10):
        self.method = method
        self.n_clusters = n_clusters

    def fit_predict(self, embeddings):
        X = embeddings.cpu().numpy()
        if self.method == 'kmeans':
            from sklearn.cluster import KMeans
            return torch.tensor(
                KMeans(n_clusters=self.n_clusters, random_state=42,
                       n_init=10).fit_predict(X),
            )
        from sklearn.cluster import HDBSCAN
        return torch.tensor(HDBSCAN(min_cluster_size=5).fit_predict(X))


# ── Instantiate ──────────────────────────────────────────────────────
m = CFG['model']
model = STIXBert(
    node_types=data.node_types, edge_types=data.edge_types,
    input_dim=m['input_dim'], hidden_dim=m['hidden_dim'],
    num_heads=m['num_heads'], num_layers=m['num_layers'],
    dropout=m['dropout'],
)
total_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {total_params:,} | '
      f'~{total_params * 4 / 1024 / 1024:.1f} MB (fp32)')

## 5. Self-Supervised Pre-training

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────
def create_node_masks(data, mask_ratio=0.15):
    masks = {}
    for nt in data.node_types:
        n = data[nt].num_nodes
        num_mask = max(1, int(n * mask_ratio))
        mask = torch.zeros(n, dtype=torch.bool)
        mask[random.sample(range(n), num_mask)] = True
        masks[nt] = mask
    return masks

def sample_negative_edges(data, edge_type, num_neg):
    st, _, dt = edge_type
    ns, nd = data[st].num_nodes, data[dt].num_nodes
    existing = set()
    if edge_type in data.edge_types:
        ei = data[edge_type].edge_index
        for i in range(ei.shape[1]):
            existing.add((ei[0, i].item(), ei[1, i].item()))
    neg_s, neg_d = [], []
    for _ in range(num_neg * 10):
        if len(neg_s) >= num_neg:
            break
        s, d = random.randint(0, ns - 1), random.randint(0, nd - 1)
        if (s, d) not in existing:
            neg_s.append(s)
            neg_d.append(d)
    return torch.tensor(neg_s), torch.tensor(neg_d)

def sample_temporal_pairs(nodes, num_pairs):
    def parse_ts(obj):
        for f in ['first_seen', 'created', 'first_observed', 'valid_from']:
            ts = obj.get(f)
            if ts:
                try:
                    return datetime.fromisoformat(ts.replace('Z', '+00:00'))
                except Exception:
                    pass
        return None
    ts = [(i, parse_ts(n)) for i, n in enumerate(nodes)]
    ts = [(i, t) for i, t in ts if t]
    if len(ts) < 2:
        return []
    pairs = []
    for _ in range(num_pairs):
        a, b = random.sample(ts, 2)
        pairs.append((a[0], b[0], 1.0 if a[1] < b[1] else 0.0))
    return pairs

# ── Training loop ────────────────────────────────────────────────────
tc = CFG['training']
model.to(DEVICE)
optimizer = AdamW(model.parameters(), lr=tc['lr'],
                  weight_decay=tc['weight_decay'])
lw = tc['loss_weights']
history = []

for epoch in range(1, tc['num_epochs'] + 1):
    model.train()
    x_dict = {nt: data[nt].x.to(DEVICE) for nt in data.node_types}
    ei_dict = {et: data[et].edge_index.to(DEVICE) for et in data.edge_types}
    masks = create_node_masks(data, tc['mask_ratio'])

    orig = {nt: x.clone() for nt, x in x_dict.items()}
    masked_x = {}
    for nt, x in x_dict.items():
        xm = x.clone()
        if nt in masks:
            xm[masks[nt].to(DEVICE)] = 0.0
        masked_x[nt] = xm

    emb = model(masked_x, ei_dict)
    masks_d = {nt: m.to(DEVICE) for nt, m in masks.items()}

    # Loss 1: masked node prediction
    loss_mask = model.masked_node_head(emb, orig, masks_d)

    # Loss 2: link prediction
    loss_link = torch.tensor(0.0, device=DEVICE)
    link_count = 0
    for et in data.edge_types:
        ei = ei_dict[et]
        if ei.shape[1] == 0:
            continue
        st, _, dt = et
        pos_src = emb[st][ei[0]]
        pos_dst = emb[dt][ei[1]]
        n_neg = min(ei.shape[1], 1000)
        ns, nd = sample_negative_edges(data, et, n_neg)
        neg_src = emb[st][ns.to(DEVICE)]
        neg_dst = emb[dt][nd.to(DEVICE)]
        logits = torch.cat([
            model.link_pred_head(pos_src, pos_dst),
            model.link_pred_head(neg_src, neg_dst),
        ])
        labels = torch.cat([
            torch.ones(ei.shape[1], device=DEVICE),
            torch.zeros(len(ns), device=DEVICE),
        ])
        loss_link += F.binary_cross_entropy_with_logits(logits, labels)
        link_count += 1
    if link_count > 0:
        loss_link /= link_count

    # Loss 3: temporal ordering
    loss_temp = torch.tensor(0.0, device=DEVICE)
    temp_count = 0
    for nt, nodes in builder.nodes_by_type.items():
        if nt not in emb:
            continue
        pairs = sample_temporal_pairs(nodes, min(len(nodes), 256))
        if not pairs:
            continue
        ia = torch.tensor([p[0] for p in pairs], device=DEVICE)
        ib = torch.tensor([p[1] for p in pairs], device=DEVICE)
        lb = torch.tensor(
            [p[2] for p in pairs], dtype=torch.float, device=DEVICE,
        )
        pred = model.temporal_head(emb[nt][ia], emb[nt][ib])
        loss_temp += F.binary_cross_entropy_with_logits(pred, lb)
        temp_count += 1
    if temp_count > 0:
        loss_temp /= temp_count

    total_loss = (lw['masked_node'] * loss_mask
                  + lw['link_pred'] * loss_link
                  + lw['temporal'] * loss_temp)

    optimizer.zero_grad()
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(
        model.parameters(), max_norm=tc['grad_clip_norm'],
    )
    optimizer.step()

    losses = {
        'total': total_loss.item(),
        'masked_node': loss_mask.item(),
        'link_pred': loss_link.item(),
        'temporal': loss_temp.item(),
    }
    history.append(losses)

    if epoch % tc['log_every'] == 0:
        print(
            f"Epoch {epoch:3d}/{tc['num_epochs']} | "
            f"Total: {losses['total']:.4f} | "
            f"Mask: {losses['masked_node']:.4f} | "
            f"Link: {losses['link_pred']:.4f} | "
            f"Temp: {losses['temporal']:.4f}"
        )

    if epoch % tc['checkpoint_every'] == 0:
        ckpt = PATHS['checkpoints'] / f'stixbert_epoch_{epoch}.pt'
        torch.save({
            'epoch': epoch,
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
        }, ckpt)

# Final checkpoint
torch.save(
    {'epoch': epoch, 'model': model.state_dict(),
     'optimizer': optimizer.state_dict()},
    PATHS['checkpoints'] / 'stixbert_final.pt',
)
print(f"\nDone! Checkpoints saved to {PATHS['checkpoints']}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, key in zip(axes.flatten(),
                   ['total', 'masked_node', 'link_pred', 'temporal']):
    ax.plot([h[key] for h in history])
    ax.set_title(f'{key} loss')
    ax.set_xlabel('Epoch')
    ax.grid(True, alpha=0.3)
plt.suptitle('STIXBert Pre-training Losses')
plt.tight_layout()
plt.savefig(
    str(PATHS['results'] / 'training_losses.png'),
    dpi=150, bbox_inches='tight',
)
plt.show()

## 6. Demo 1 — Campaign Clustering (UMAP)

In [ ]:
from umap import UMAP

embeddings = model.get_embeddings(
    {nt: data[nt].x.to(DEVICE) for nt in data.node_types},
    {et: data[et].edge_index.to(DEVICE) for et in data.edge_types},
)

if 'malware' in embeddings:
    mw_emb = embeddings['malware']
    clusterer = CampaignClusterer(method='hdbscan')
    cluster_labels = clusterer.fit_predict(mw_emb)
    n_clusters = len(set(cluster_labels.numpy())) - (
        1 if -1 in cluster_labels.numpy() else 0
    )
    print(f'Discovered {n_clusters} malware clusters')

    X2 = UMAP(n_components=2, random_state=42).fit_transform(
        mw_emb.cpu().numpy(),
    )
    fig, ax = plt.subplots(figsize=(12, 8))
    y = cluster_labels.numpy()
    for l in sorted(set(y)):
        m = y == l
        ax.scatter(X2[m, 0], X2[m, 1], label=f'Cluster {l}',
                   alpha=0.6, s=20)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    ax.set_title('STIXBert: Malware Campaign Clustering (UMAP)')
    plt.tight_layout()
    plt.savefig(
        str(PATHS['results'] / 'campaign_clusters.png'),
        dpi=150, bbox_inches='tight',
    )
    plt.show()

## 7. Demo 2 — ATT&CK Classification + Label Efficiency

In [ ]:
def label_efficiency_experiment(encoder, clf_class, data, labels,
                                node_type, fractions, pretrained=True,
                                num_trials=3):
    encoder.eval()
    with torch.no_grad():
        xd = {nt: data[nt].x.to(DEVICE) for nt in data.node_types}
        eid = {et: data[et].edge_index.to(DEVICE) for et in data.edge_types}
        emb = encoder(xd, eid)[node_type]
    nc = labels.max().item() + 1
    results = {}
    for frac in fractions:
        accs = []
        n = labels.shape[0]
        for _ in range(num_trials):
            nt_ = max(10, int(n * frac))
            idx = list(range(n))
            random.shuffle(idx)
            tr, te = idx[:nt_], idx[nt_:] or idx
            clf = clf_class(emb.shape[1], nc).to(DEVICE)
            opt = torch.optim.Adam(clf.parameters(), lr=1e-3)
            clf.train()
            for __ in range(100):
                loss = nn.CrossEntropyLoss()(
                    clf(emb[tr]), labels[tr].to(DEVICE),
                )
                opt.zero_grad()
                loss.backward()
                opt.step()
            clf.eval()
            with torch.no_grad():
                pred = clf(emb[te]).argmax(1)
                accs.append(
                    (pred == labels[te].to(DEVICE)).float().mean().item(),
                )
        results[frac] = sum(accs) / len(accs)
        tag = 'pretrained' if pretrained else 'scratch'
        print(f'  [{tag}] {frac:.0%} labels -> acc={results[frac]:.3f}')
    return results

if 'attack_pattern' in embeddings:
    attack_nodes = builder.nodes_by_type.get('attack_pattern', [])
    tactic_map, labels_list = {}, []
    for node in attack_nodes:
        phases = node.get('kill_chain_phases', [])
        tactic = (phases[0].get('phase_name', 'unknown')
                  if phases else 'unknown')
        if tactic not in tactic_map:
            tactic_map[tactic] = len(tactic_map)
        labels_list.append(tactic_map[tactic])
    labels = torch.tensor(labels_list)
    print(f'ATT&CK tactics: {len(tactic_map)} classes')

    fracs = CFG['finetune']['label_efficiency_fractions']

    print('\n--- Pre-trained ---')
    res_pre = label_efficiency_experiment(
        model.encoder, ATTACKClassifier, data, labels,
        'attack_pattern', fracs, pretrained=True,
    )

    rand_model = STIXBert(
        node_types=data.node_types, edge_types=data.edge_types,
        input_dim=128, hidden_dim=128, num_heads=4, num_layers=4,
    ).to(DEVICE)

    print('\n--- Random init ---')
    res_rand = label_efficiency_experiment(
        rand_model.encoder, ATTACKClassifier, data, labels,
        'attack_pattern', fracs, pretrained=False,
    )

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(fracs, [res_pre[f] for f in fracs], 'o-',
            label='Pre-trained (STIXBert)')
    ax.plot(fracs, [res_rand[f] for f in fracs], 's--',
            label='Random init')
    ax.set_xlabel('Fraction of labeled data')
    ax.set_ylabel('Accuracy')
    ax.set_title('Label Efficiency: Pre-trained vs Random Init')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(
        str(PATHS['results'] / 'label_efficiency.png'), dpi=150,
    )
    plt.show()

## 8. Demo 3 — Cross-Feed Deduplication Heatmap

In [ ]:
indicator_nodes = builder.nodes_by_type.get('indicator', [])
if indicator_nodes and 'indicator' in embeddings:
    ind_emb = embeddings['indicator']
    by_feed = defaultdict(list)
    for i, node in enumerate(indicator_nodes):
        by_feed[node.get('_source', 'unknown')].append(i)

    feed_emb = {
        f: ind_emb[torch.tensor(idx)]
        for f, idx in by_feed.items() if len(idx) >= 10
    }

    if len(feed_emb) >= 2:
        names = list(feed_emb.keys())
        n = len(names)
        mat = np.zeros((n, n))
        for i in range(n):
            for j in range(n):
                if i == j:
                    mat[i, j] = 1.0
                    continue
                ei = feed_emb[names[i]]
                ej = feed_emb[names[j]]
                ein = ei / ei.norm(dim=1, keepdim=True)
                ejn = ej / ej.norm(dim=1, keepdim=True)
                sim = torch.mm(ein, ejn.t())
                mat[i, j] = (
                    (sim.max(dim=1).values > 0.9).float().mean().item()
                )

        fig, ax = plt.subplots(figsize=(8, 6))
        im = ax.imshow(mat, cmap='YlOrRd', vmin=0, vmax=1)
        ax.set_xticks(range(n))
        ax.set_yticks(range(n))
        ax.set_xticklabels(names, rotation=45, ha='right')
        ax.set_yticklabels(names)
        for i in range(n):
            for j in range(n):
                ax.text(j, i, f'{mat[i, j]:.0%}',
                        ha='center', va='center', fontsize=9)
        ax.set_title('Cross-Feed Overlap (cosine sim > 0.9)')
        fig.colorbar(im, ax=ax)
        plt.tight_layout()
        plt.savefig(
            str(PATHS['results'] / 'cross_feed_overlap.png'),
            dpi=150, bbox_inches='tight',
        )
        plt.show()
    else:
        print('Need >= 2 feeds with >= 10 indicators')
else:
    print('No indicator nodes — expected if include_scos=False')

## 9. Demo 4 — Predictive Link Prediction

In [ ]:
if 'malware' in embeddings and 'attack_pattern' in embeddings:
    mw_emb = embeddings['malware']
    ap_emb = embeddings['attack_pattern']
    mw_nodes = builder.nodes_by_type.get('malware', [])
    ap_nodes = builder.nodes_by_type.get('attack_pattern', [])

    if mw_nodes:
        sample_name = mw_nodes[0].get('name', 'Unknown')
        sample_emb = mw_emb[0].unsqueeze(0)
        scores = model.link_pred_head(
            sample_emb.expand(ap_emb.shape[0], -1), ap_emb,
        )
        probs = torch.sigmoid(scores)
        top_scores, top_idx = probs.topk(10)

        print(f'\nMalware: {sample_name}')
        print('Top 10 predicted ATT&CK techniques:')
        for i, (score, idx) in enumerate(zip(top_scores, top_idx)):
            ap = ap_nodes[idx.item()]
            tid = ''
            for ref in ap.get('external_references', []):
                if ref.get('source_name') == 'mitre-attack':
                    tid = ref.get('external_id', '')
                    break
            print(f'  {i+1}. {tid} — {ap.get("name", "?")} '
                  f' (score: {score.item():.3f})')

## 10. Save Final Results

In [ ]:
with open(PATHS['results'] / 'training_history.json', 'w') as f:
    json.dump(history, f, indent=2)

torch.save(
    {nt: e.cpu() for nt, e in embeddings.items()},
    PATHS['eval'] / 'embeddings.pt',
)

print('All data persisted to Google Drive:')
for key, path in sorted(PATHS.items()):
    n_files = len(list(path.glob('*'))) if path.exists() else 0
    print(f'  {key:20s} -> {path}  ({n_files} files)')

## 11. Push Model to HuggingFace Hub

`HF_TOKEN` is read from **Colab Secrets** (Key icon in sidebar).

In [ ]:
import shutil
from huggingface_hub import HfApi, create_repo

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded from Colab Secrets')
except (ImportError, userdata.SecretNotFoundError):
    raise RuntimeError(
        'HF_TOKEN not found. In Colab: Key icon -> Add HF_TOKEN.\n'
        'Get a token at: https://huggingface.co/settings/tokens',
    )

hf = CFG['huggingface']
repo_id = hf['repo_id']
upload_dir = Path('hf_upload')
upload_dir.mkdir(exist_ok=True)

shutil.copy(
    PATHS['checkpoints'] / 'stixbert_final.pt',
    upload_dir / 'stixbert_final.pt',
)
with open(upload_dir / 'config.json', 'w') as f:
    json.dump(CFG, f, indent=2)
with open(upload_dir / 'graph_metadata.json', 'w') as f:
    json.dump({
        'node_types': data.node_types,
        'edge_types': [list(et) for et in data.edge_types],
        'graph_stats': stats,
        'model_config': CFG['model'],
    }, f, indent=2)
shutil.copy(
    str(PATHS['results'] / 'training_history.json'),
    upload_dir / 'training_history.json',
)

mc = CFG['model']
tags_yaml = '\n'.join(f'- {t}' for t in hf['tags'])
model_card = f"""---
language: en
license: apache-2.0
library_name: pytorch
pipeline_tag: graph-ml
tags:
{tags_yaml}
---

# STIXBert — Self-Supervised STIX Graph Foundation Model

HGT pre-trained on STIX 2.1 threat intelligence graphs.

| Property | Value |
|----------|-------|
| Architecture | HGT ({mc['num_layers']} layers, {mc['num_heads']} heads) |
| Hidden dim | {mc['hidden_dim']} |
| Parameters | {total_params:,} |
| Text encoder | {mc['text_model']} |

## Data Sources
- MITRE ATT&CK (Enterprise, Mobile, ICS)
- ThreatFox (abuse.ch)
- DigitalSide Threat-Intel
"""
(upload_dir / 'README.md').write_text(model_card)

api = HfApi(token=hf_token)
create_repo(
    repo_id=repo_id, private=hf.get('private', True),
    repo_type='model', exist_ok=True, token=hf_token,
)
api.upload_folder(
    folder_path=str(upload_dir), repo_id=repo_id,
    commit_message='STIXBert PoV checkpoint',
)
print(f'\nPushed to: https://huggingface.co/{repo_id}')
for f in sorted(upload_dir.iterdir()):
    print(f'  {f.name}  ({f.stat().st_size / 1024 / 1024:.1f} MB)')
shutil.rmtree(upload_dir)